# 10장. 분류 분석으로 주문 취소 여부 예측하기

이 노트북은 완료 주문과 취소 주문만 사용해 `is_cancelled`를 예측하는 이진 분류 실습입니다.

## 학습 목표

- `completed=0`, `cancelled=1`로 타깃 범위를 명확하게 정의합니다.
- `refunded`와 기타 상태를 이진 분류 대상에서 제외합니다.
- 병합의 키 관계, 행 수, 미매칭을 검증합니다.
- train, validation, test를 분리합니다.
- Dummy 기준 모델, Logistic Regression, Random Forest를 비교합니다.
- validation에서 모델과 임계값을 선택하고 test는 최종 평가에 한 번만 사용합니다.
- 예측 결과에서 고객명 등 불필요한 개인정보를 제외합니다.


## 1. 프로젝트 루트와 실행 환경 설정

전처리 파일이 없다면 프로젝트 루트에서 먼저 다음 명령을 실행하세요.

```bash
python scripts/preprocess_data.py
```


In [ ]:
from pathlib import Path
import sys

import pandas as pd


def find_project_root(start_path):
    start_path = Path(start_path).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / 'requirements.txt').exists() and (candidate / 'scripts').exists():
            return candidate
    raise FileNotFoundError('프로젝트 루트 폴더를 찾을 수 없습니다.')


PROJECT_ROOT = find_project_root(Path.cwd())
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('Python 실행 파일:', sys.executable)
print('프로젝트 루트:', PROJECT_ROOT)
print('전처리 데이터 폴더:', PROCESSED_DIR)
print('보고서 폴더:', REPORT_DIR)


## 2. 전처리 데이터 확인과 불러오기

이 장에서는 `customers_clean.csv`, `orders_clean.csv`, `order_items_clean.csv`를 사용합니다.


In [ ]:
required_files = [
    PROCESSED_DIR / 'customers_clean.csv',
    PROCESSED_DIR / 'orders_clean.csv',
    PROCESSED_DIR / 'order_items_clean.csv',
]

missing_files = [path for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(
        '전처리 파일이 없습니다: '
        + ', '.join(str(path) for path in missing_files)
        + '. 먼저 python scripts/preprocess_data.py를 실행하세요.'
    )

customers = pd.read_csv(PROCESSED_DIR / 'customers_clean.csv')
orders = pd.read_csv(PROCESSED_DIR / 'orders_clean.csv')
order_items = pd.read_csv(PROCESSED_DIR / 'order_items_clean.csv')

print('customers:', customers.shape, customers.columns.tolist())
print('orders:', orders.shape, orders.columns.tolist())
print('order_items:', order_items.shape, order_items.columns.tolist())


## 3. 주문 상태와 타깃 범위 확인

이번 실습은 완료 주문과 취소 주문만 비교합니다.

- `completed` → 0
- `cancelled` → 1
- `refunded`와 기타 상태 → 학습 대상에서 제외


In [ ]:
status_summary = (
    orders['order_status']
    .value_counts(dropna=False)
    .rename_axis('order_status')
    .reset_index(name='order_count')
)
status_summary


## 4. 분류 데이터와 병합 검증표 생성

공통 함수는 주문 특징 집계, 안전한 병합, 시간 순서 점검, 타깃 생성까지 수행합니다.


In [ ]:
from src.classification import (
    build_classification_dataset,
    target_distribution,
)

(
    model_data,
    numeric_features,
    categorical_features,
    merge_checks,
    data_quality_checks,
) = build_classification_dataset(
    customers=customers,
    orders=orders,
    order_items=order_items,
)

target_dist = target_distribution(model_data)

print('모델링 데이터:', model_data.shape)
print('숫자형 feature:', numeric_features)
print('범주형 feature:', categorical_features)
display(target_dist)
display(merge_checks)
display(data_quality_checks)


## 5. 데이터 누수 확인

`order_status`와 `is_cancelled`는 입력 feature에 포함되면 안 됩니다.


In [ ]:
features = numeric_features + categorical_features
leakage_columns = {'order_status', 'is_cancelled'}
leakage_found = sorted(leakage_columns.intersection(features))

if leakage_found:
    raise ValueError(f'데이터 누수 위험 컬럼이 포함되었습니다: {leakage_found}')

print('사용 feature:', features)
print('데이터 누수 컬럼 없음')


## 6. train, validation, test 분리

모델과 임계값은 validation에서 선택하고, test는 최종 평가에 한 번만 사용합니다.


In [ ]:
from src.classification import (
    split_train_validation_test,
    build_split_summary,
)

(
    X_train,
    X_valid,
    X_test,
    y_train,
    y_valid,
    y_test,
    features,
) = split_train_validation_test(
    model_data=model_data,
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    random_state=42,
)

split_summary = build_split_summary(y_train, y_valid, y_test)

print('X_train:', X_train.shape)
print('X_valid:', X_valid.shape)
print('X_test:', X_test.shape)
display(split_summary)


## 7. 기준 모델과 학습 모델 비교

가장 많은 클래스만 예측하는 Dummy 모델을 기준으로 Logistic Regression과 Random Forest의 validation 성능을 비교합니다.


In [ ]:
from src.classification import train_and_compare_on_validation

(
    models,
    validation_comparison,
    validation_predictions,
    validation_probabilities,
) = train_and_compare_on_validation(
    X_train=X_train,
    X_valid=X_valid,
    y_train=y_train,
    y_valid=y_valid,
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    random_state=42,
)

validation_comparison


## 8. validation에서 모델과 임계값 선택

Dummy 모델은 기준선이므로 실제 선택 후보에서는 제외합니다. validation의 f1-score를 우선하고 recall과 precision을 함께 확인합니다.


In [ ]:
from src.classification import threshold_metrics, choose_threshold

non_dummy_comparison = validation_comparison[
    validation_comparison['model'] != 'Dummy Most Frequent'
]

selected_model_name = str(non_dummy_comparison.iloc[0]['model'])
selected_model = models[selected_model_name]
validation_proba = validation_probabilities[selected_model_name]

threshold_df = threshold_metrics(y_valid, validation_proba)
selected_threshold = choose_threshold(threshold_df)

print('선택 모델:', selected_model_name)
print('선택 임계값:', selected_threshold)
display(threshold_df.sort_values(['f1', 'recall'], ascending=False).head(10))


## 9. test 데이터 최종 평가

이제 선택이 끝났으므로 test 데이터에서 최종 성능을 한 번 확인합니다.


In [ ]:
from src.classification import (
    final_test_evaluation,
    confusion_matrix_dataframe,
    classification_report_dataframe,
)

y_pred_test, y_proba_test, test_metrics = final_test_evaluation(
    selected_model,
    X_test,
    y_test,
    threshold=selected_threshold,
)

confusion_df = confusion_matrix_dataframe(y_test, y_pred_test)
classification_report_df = classification_report_dataframe(y_test, y_pred_test)

display(test_metrics)
display(confusion_df)
display(classification_report_df)


혼동행렬은 다음처럼 읽습니다.

| 구분 | 의미 |
|---|---|
| True Negative | 실제 완료 주문을 완료로 예측 |
| False Positive | 실제 완료 주문을 취소로 잘못 예측 |
| False Negative | 실제 취소 주문을 완료로 잘못 예측 |
| True Positive | 실제 취소 주문을 취소로 예측 |


## 10. 개인정보를 제외한 예측 결과 생성

고객명, 이메일, 주소 같은 불필요한 식별정보는 결과 파일에 저장하지 않습니다.


In [ ]:
from src.classification import create_prediction_result

prediction_result = create_prediction_result(
    source_index=X_test.index,
    y_test=y_test,
    y_pred=y_pred_test,
    y_proba=y_proba_test,
    model_name=selected_model_name,
    threshold=selected_threshold,
)

prediction_result.head()


## 11. LLM 코드 검토 체크리스트


In [ ]:
from src.classification import build_classification_checklist

classification_checklist = build_classification_checklist()
classification_checklist


## 12. 전체 파이프라인 실행과 산출물 저장

위 과정을 한 번에 다시 실행해 CSV와 Markdown 보고서를 생성합니다.


In [ ]:
from src.classification import run_classification_analysis

classification_result = run_classification_analysis(
    processed_dir=PROCESSED_DIR,
    report_dir=REPORT_DIR,
    random_state=42,
)

print('선택 모델:', classification_result['selected_model_name'])
print('선택 임계값:', classification_result['selected_threshold'])
display(classification_result['test_metrics'])

for name, path in classification_result['output_paths'].items():
    print(name, path, 'OK' if path.exists() else 'MISSING')


## 13. LLM 검토 프롬프트 예시

```text
온라인 쇼핑몰 주문 취소 분류 코드를 검토해 주세요.

타깃 기준:
- completed=0
- cancelled=1
- refunded와 기타 상태는 제외

검토 항목:
1. order_status와 is_cancelled가 feature에서 제외되었는가?
2. 병합에 validate와 indicator가 사용되었는가?
3. train, validation, test가 분리되었는가?
4. 모델과 임계값은 validation에서 선택했는가?
5. test는 최종 평가에 한 번만 사용했는가?
6. Dummy 기준 모델보다 나은가?
7. accuracy, precision, recall, f1을 함께 확인했는가?
8. 예측 결과에 개인정보가 포함되지 않았는가?
9. 모델 결과를 취소 원인으로 단정하지 않았는가?

필수 수정과 권장 개선을 구분해 설명해 주세요.
```


## 14. 정리

이번 장에서는 완료 주문과 취소 주문을 명확하게 구분하고, 병합 검증, 데이터 누수 방지, train/validation/test 분리, 기준 모델 비교, validation 기반 모델·임계값 선택, 최종 test 평가를 수행했습니다.

다음 장에서는 LLM을 활용해 분석 질문과 프롬프트를 더 구체적으로 만드는 방법으로 이어집니다.
